## Jupyter kernel from the pyQuil container (OnDemand)

This notebook registers a Jupyter kernel that runs Python **inside** the
`pyquil.sif` Apptainer container, so the other notebooks in this folder get
pyQuil 4, forest-benchmarking and Rigetti's `quilc` compiler and `qvm`
simulator (Forest SDK 2.23.0) without a local virtual environment:

| kernel | Apptainer flags | use |
|---|---|---|
| **Python (pyQuil container)** | `--cleanenv --bind /ocean` | any CPU node |

Run it **once**, with the default **Python 3 (ipykernel)** kernel selected in
the top-right corner. The kernel is saved in
`~/.local/share/jupyter/kernels/` and stays available in later sessions.

The container was built from `pyquil.def` in this repository
(`apptainer build --fakeroot pyquil.sif pyquil.def`).

##### 1. Check that the container starts and has `ipykernel`, `pyquil`, `quilc` and `qvm`

In [1]:
# `!` runs a shell command.  `apptainer exec` starts the container and runs
# a command inside it; ipykernel is what lets Jupyter talk to that Python.
SIF = "/ocean/projects/pscstaff/nnepal/Rigetti/pyquil.sif"   # change if the image is elsewhere
!apptainer exec --cleanenv {SIF} python -c "import ipykernel, pyquil, forest.benchmarking; print('ipykernel', ipykernel.__version__, '| pyquil', pyquil.__version__)"
!apptainer exec --cleanenv {SIF} quilc --version
!apptainer exec --cleanenv {SIF} qvm --version

ipykernel 7.4.0 | pyquil 4.22.0
1.23.0 [e6c0939]
1.17.1 [cf3f91f]


##### 2. Register the kernel

In [2]:
import json, os, shutil   # standard library

sif    = SIF
runner = shutil.which("apptainer")           # full path of the apptainer command
assert runner and os.path.exists(sif), "apptainer or the container image is missing"

name, display = "pyquil-container", "Python (pyQuil container)"
# Jupyter finds user kernels in ~/.local/share/jupyter/kernels/<name>/kernel.json
kdir = os.path.expanduser(f"~/.local/share/jupyter/kernels/{name}")
os.makedirs(kdir, exist_ok=True)
# The command Jupyter runs to start the kernel:
#   --cleanenv            do not pass the host environment (OnDemand's Python
#                         paths, LD_LIBRARY_PATH) into the container
#   --bind /ocean         make project folders readable/writable
#   PYTHONNOUSERSITE=1    ignore packages in ~/.local, use the container's only
#   {connection_file}     Jupyter fills this in with the kernel's port file
argv = [runner, "exec", "--cleanenv",
        "--bind", "/ocean",
        "--env", "PYTHONNOUSERSITE=1",
        sif, "python", "-m", "ipykernel_launcher", "-f", "{connection_file}"]
with open(f"{kdir}/kernel.json", "w") as f:
    json.dump({"argv": argv, "display_name": display, "language": "python"}, f, indent=2)
print("created", name, "->", kdir)

created pyquil-container -> /jet/home/nnepal/.local/share/jupyter/kernels/pyquil-container


From now on, select **Python (pyQuil container)** in the top-right corner
(refresh the page if it is not listed yet).

Inside this kernel `quilc` and `qvm` are already on `PATH`, and the container
provides `libzmq.so` and BLAS/LAPACK, so the `pyzmq<26` / `LD_LIBRARY_PATH`
steps and `module load openblas` from the README are **not** needed.

> **Caution:** pyQuil reaches the servers at `127.0.0.1`, so they must run on
> the same node as the kernel. Starting them from the kernel (cell below)
> guarantees that. `--cleanenv` also removes the Slurm variables.

##### 3. Check the new kernel

Switch to **Python (pyQuil container)**, then run the cells below. The first
starts `qvm` (port 5000) and `quilc` (port 5555) inside the container in the
background; the second runs a Bell state and should print 10 rows that are each
`[0 0]` or `[1 1]`.

In [20]:
import subprocess, socket, time, sys

def port_open(port):
    with socket.socket() as s:
        return s.connect_ex(("127.0.0.1", port)) == 0

for cmd, port in (("qvm -S", 5000), ("quilc -P -S", 5555)):
    name = cmd.split()[0]
    if port_open(port):
        print(f"{name}: port {port} already in use, assuming it is running")
        continue
    subprocess.Popen(cmd.split(), stdout=open(f"{name}.log", "w"),
                     stderr=subprocess.STDOUT, start_new_session=True)
    for _ in range(60):
        if port_open(port):
            break
        time.sleep(1)
    print(f"{name}: {'running' if port_open(port) else 'NOT running, see ' + name + '.log'} (port {port})")
print(sys.executable)

qvm: running (port 5000)
quilc: running (port 5555)
/opt/venv/bin/python


##### Which servers are running?

Run this at any time to see what is listening on the two ports pyQuil uses on
this node: **qvm** on 5000 and **quilc** on 5555. For each port it reports
the process that owns it (PID, user, uptime, command and parent) and asks the
server for its version, which confirms that it is really qvm or quilc and that
it responds. If the parent is an `ipykernel_launcher`, the servers were started
by a Jupyter kernel (the cell above).

> **Caution:** the servers belong to the node they run on. A new OnDemand
> session on another node starts with no servers, and servers left from an old
> kernel on this node keep running until you stop them (last cells).

In [23]:
import glob, os, pwd, socket, time
from qcs_sdk.qvm import QVMClient, api as qvm_api
from qcs_sdk.compiler.quilc import QuilcClient, get_version_info as quilc_version

SERVERS = {5000: "qvm", 5555: "quilc"}      # the ports pyQuil connects to on 127.0.0.1

def listening_inodes(port):
    """Socket inodes LISTENing on `port`, from the kernel's TCP tables.
    Each /proc/net/tcp row has 'local_address' as HEXIP:HEXPORT, state 0A = LISTEN,
    and the socket's inode number in column 10."""
    inodes = set()
    for table in ("/proc/net/tcp", "/proc/net/tcp6"):
        try:
            rows = open(table).read().splitlines()[1:]
        except OSError:
            continue
        for row in rows:
            f = row.split()
            if int(f[1].split(":")[1], 16) == port and f[3] == "0A":
                inodes.add(f[9])
    return inodes

def owner_pids(inodes):
    """Processes holding one of those sockets.  /proc/<pid>/fd/* links read
    'socket:[inode]'; only your own processes' fd lists are readable, so a
    server run by another user is reported as 'not visible'."""
    pids = set()
    for fd in glob.glob("/proc/[0-9]*/fd/*"):
        try:
            link = os.readlink(fd)
        except OSError:
            continue
        if link.startswith("socket:[") and link[8:-1] in inodes:
            pids.add(int(fd.split("/")[2]))
    return sorted(pids)

def describe(pid):
    """User, parent PID, uptime and command line of a process, from /proc."""
    status = dict(l.split(":", 1) for l in open(f"/proc/{pid}/status").read().splitlines() if ":" in l)
    user = pwd.getpwuid(int(status["Uid"].split()[0])).pw_name
    ppid = int(status["PPid"])
    cmd = open(f"/proc/{pid}/cmdline").read().replace("\0", " ").strip()
    # field 22 of /proc/<pid>/stat is the start time in clock ticks since boot
    start_ticks = int(open(f"/proc/{pid}/stat").read().rsplit(")", 1)[1].split()[19])
    uptime = float(open("/proc/uptime").read().split()[0])
    age = uptime - start_ticks / os.sysconf("SC_CLK_TCK")
    parent = open(f"/proc/{ppid}/cmdline").read().replace("\0", " ").strip()[:70] if os.path.exists(f"/proc/{ppid}") else "?"
    return f"PID {pid}, user {user}, running {age/60:.1f} min, command '{cmd}'\n        parent {ppid}: {parent}"

def version(name):
    """Ask the server itself: proves it is qvm/quilc and that it answers."""
    if name == "qvm":
        return qvm_api.get_version_info(QVMClient.new_http("http://127.0.0.1:5000"))
    return quilc_version(QuilcClient.new_rpcq("tcp://127.0.0.1:5555"))

print(f"node {socket.gethostname()}")
for port, name in SERVERS.items():
    inodes = listening_inodes(port)
    if not inodes:
        print(f"{name:5s} port {port}: nothing listening -> start it with the cell above")
        continue
    pids = owner_pids(inodes)
    who = "\n      ".join(describe(p) for p in pids) if pids else "owner not visible (another user's process?)"
    try:
        ver = f"answers, version {version(name)}"
    except Exception as exc:
        ver = f"does NOT answer as {name} ({type(exc).__name__}: {exc})"
    print(f"{name:5s} port {port}: listening, {ver}\n      {who}")

node r040.ib.bridges2.psc.edu
qvm   port 5000: nothing listening -> start it with the cell above
quilc port 5555: nothing listening -> start it with the cell above


##### Deprecation warnings from pyQuil 4.22

The container installs pyQuil 4.22.0. In that release every `get_qc("...-qvm")`
call prints three `PyQuilDeprecationWarning` notices, one each for `get_qc`,
`QVM` and `QVMCompiler`:

```
/tmp/ipykernel_79063/4034777084.py:1: PyQuilDeprecationWarning: Getting a QVM-backed quantum computer from get_qc (here '1q-qvm') is deprecated and will be removed in pyQuil v5, along with the QVM, PyQVM and QVMCompiler. pyQuil v4 already includes the Quax-based simulators and noise model that replace it, as private, experimental modules (see https://pyquil-docs.rigetti.com/en/4.22.0/simulation_architecture.html), so we recommend continuing to use this API until you upgrade to pyQuil v5. -- Deprecated since version 4.22.0.
  qc = get_qc('1q-qvm')
/opt/venv/lib/python3.12/site-packages/pyquil/api/_quantum_computer.py:522: PyQuilDeprecationWarning: Call to deprecated class QVM. (To be removed in pyQuil v5, where TrajectorySimulator samples program outcomes instead. pyQuil v4 already includes the Quax-based simulators and noise model that replace it, as private, experimental modules (see https://pyquil-docs.rigetti.com/en/4.22.0/simulation_architecture.html), so we recommend continuing to use this API until you upgrade to pyQuil v5.) -- Deprecated since version 4.22.0.
  return QVM(noise_model=noise_model, timeout=execution_timeout, client=qvm_client)
/opt/venv/lib/python3.12/site-packages/pyquil/api/_quantum_computer.py:565: PyQuilDeprecationWarning: Call to deprecated class QVMCompiler. (To be removed in pyQuil v5 along with the QVM; the Quax-based simulators that replace the QVM run programs without compiling them. pyQuil v4 already includes the Quax-based simulators and noise model that replace it, as private, experimental modules (see https://pyquil-docs.rigetti.com/en/4.22.0/simulation_architecture.html), so we recommend continuing to use this API until you upgrade to pyQuil v5.) -- Deprecated since version 4.22.0.
  compiler=QVMCompiler(
```

They are notices, not errors: the QVM still works, and pyQuil itself recommends
using it until pyQuil v5, because the replacement simulators are private and
experimental in v4. Every notebook in this folder therefore starts its first
pyQuil cell with a **warning filter**: a rule for Python's `warnings` module
that ignores warnings of the category `PyQuilDeprecationWarning` and still
shows all others. The filter lasts until the kernel restarts. The next cell
shows it.

In [13]:
import warnings
from pyquil._deprecation import PyQuilDeprecationWarning
# Hide pyQuil 4.22's notices that the QVM is deprecated in favour of pyQuil v5
# (the QVM still works; see the explanation above).
warnings.filterwarnings("ignore", category=PyQuilDeprecationWarning)

from pyquil import Program, get_qc
from pyquil.gates import H, CNOT

qc = get_qc("2q-qvm")
p = Program(H(0), CNOT(0, 1))
p.measure_all()
p.wrap_in_numshots_loop(10)

result = qc.run(qc.compile(p))
print(result.get_register_map()["ro"])

[[1 1]
 [1 1]
 [1 1]
 [1 1]
 [1 1]
 [0 0]
 [0 0]
 [0 0]
 [1 1]
 [0 0]]


Stop the servers when you are done (or just shut the kernel down / end the
OnDemand session):

In [22]:
# -x matches the process NAME exactly (qvm, quilc).  Do not use -f "qvm -S":
# -f searches whole command lines, and the shell running this line contains
# that text too, so pkill kills the shell after qvm and never reaches quilc.
!pkill -u $(id -u) -x qvm; pkill -u $(id -u) -x quilc; echo stopped

stopped
